<div align="center">
    <h1> <b>DEPARTAMENTO DE VENTAS</b></h1>
    <h1> <b>PROYECTO DE PRONÓSTICO DE VENTAS FUTURAS</b></h1><br>
</div>
<div>
    <h3> <b>Algoritmos: Series temporales.</b></h3>
</div>

# 1. ENUNCIADO DEL PROBLEMA Y CASO PRÁCTICO.


<table>
  <tr><td>
    <img src="https://drive.google.com/uc?id=1l7bHyrjzq839zVZE06cfdDksLabCN2hg"
         alt="Fashion MNIST sprite"  width="1000">
  </td></tr>
  <tr><td align="center">
    <b>Figura 1. Prediccion de Ventas Futuras usando Series Temporales
  </td></tr>
</table>


## 1.1 Entendimiento del Negocio y Objetivo del Proyecto.

![alt text](https://drive.google.com/uc?id=179967pYtO8DZkFIiqkzKeDXpzKF13KE9)

![alt text](https://drive.google.com/uc?id=1MHVbje9FNlAbo-LOMlCvSYfpgzW7PQ4y)

## 1.2 Características del Cliente.

![alt text](https://drive.google.com/uc?id=1DC-hoQNAxfDXEn91_BTF2B8qH904N8yq)

![alt text](https://drive.google.com/uc?id=1sM7eT88kpVmNMl7wJy0Jq0X_w-r783JF)

![alt text](https://drive.google.com/uc?id=1E6qCyK6SjFany6QsXw8lUc77CSOgjTkj)

# 2. IMPORTAR LAS LIBRERÍAS Y LOS DATASETS.

## 2.1 Importe de librerías.

In [11]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import datetime

## 2.2 Cargar y Leer el Dataset.

In [59]:
# Leer el dataset
ventas = pd.read_csv("train.csv")
ventas

C:\Users\Danko\AppData\Local\Temp\ipykernel_18068\1497398823.py:2: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  ventas = pd.read_csv("train.csv")


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1
...,...,...,...,...,...,...,...,...,...
1017204,1111,2,2013-01-01,0,0,0,0,a,1
1017205,1112,2,2013-01-01,0,0,0,0,a,1
1017206,1113,2,2013-01-01,0,0,0,0,a,1
1017207,1114,2,2013-01-01,0,0,0,0,a,1


## 2.3 Leyenda.

In [13]:
ventas.columns

Index(['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo',
       'StateHoliday', 'SchoolHoliday'],
      dtype='object')

#### Casi un millón de observaciones
#### 1115 tiendas únicas 
#### Notemos que las ventas es la variable objetivo (la que intentamos predecir)

#### Id:  ID de transacción (combinación de la tienda y la fecha)
#### Store: identificador único de la tienda
#### Sales: ventas diarias, esta es la variable objetivo
#### Customers: número de clientes de un día dado
#### Open: Booleano para indicar si la tienda estaba abierta o cerrada (0 = cerrada, 1 = abierta)
#### Promo: describe si la tienda tenía algún tipo de promoción ese día o no
#### StateHoliday: indica si el día era festivo o no (a = vacaciones públicas, b = vacaciones de Pascua holiday, c = Navidades, 0 = No era festivo)
#### SchoolHoliday: indica si (Store, Date) se ve afectado por el cierre de las escuelas públicas
#### Fuente original de los datos: https://www.kaggle.com/c/rossmann-store-sales/data

## 2.4 Primeras descripciones y conclusiones.

In [15]:
ventas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 9 columns):
 #   Column         Non-Null Count    Dtype 
---  ------         --------------    ----- 
 0   Store          1017209 non-null  int64 
 1   DayOfWeek      1017209 non-null  int64 
 2   Date           1017209 non-null  object
 3   Sales          1017209 non-null  int64 
 4   Customers      1017209 non-null  int64 
 5   Open           1017209 non-null  int64 
 6   Promo          1017209 non-null  int64 
 7   StateHoliday   1017209 non-null  object
 8   SchoolHoliday  1017209 non-null  int64 
dtypes: int64(7), object(2)
memory usage: 69.8+ MB


#### Tenemos 9 variables, dos objetos: Date y StateHoliday y siete enteros.

In [16]:
ventas.describe()

,Store,DayOfWeek,Sales,Customers,Open,Promo,SchoolHoliday
count,1.017209e+06,1.017209e+06,1.017209e+06,1.017209e+06,1.017209e+06,1.017209e+06,1.017209e+06
mean,5.584297e+02,3.998341e+00,5.773819e+03,6.331459e+02,8.301067e-01,3.815145e-01,1.786467e-01
std,3.219087e+02,1.997391e+00,3.849926e+03,4.644117e+02,3.755392e-01,4.857586e-01,3.830564e-01
min,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,2.800000e+02,2.000000e+00,3.727000e+03,4.050000e+02,1.000000e+00,0.000000e+00,0.000000e+00
50%,5.580000e+02,4.000000e+00,5.744000e+03,6.090000e+02,1.000000e+00,0.000000e+00,0.000000e+00
75%,8.380000e+02,6.000000e+00,7.856000e+03,8.370000e+02,1.000000e+00,1.000000e+00,0.000000e+00
max,1.115000e+03,7.000000e+00,4.155100e+04,7.388000e+03,1.000000e+00,1.000000e+00,1.000000e+00


### Conclusiones:
##### 1.- Cantidad de ventas promedio por día = 5773 Euros
##### 2.- Ventas mínimas por día = 0, ventas máximas por día = 41551
##### 3.- Promedio de clientes = 633, número mínimo de clientes = 0, número máximo de clientes = 7388

## 2.5 Cargar segundo dataset y Leer el Dataset.

In [60]:
# Leer el dataset
tiendas = pd.read_csv("store.csv")
tiendas

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
1110,1111,a,a,1900.0,6.0,2014.0,1,31.0,2013.0,"Jan,Apr,Jul,Oct"
1111,1112,c,c,1880.0,4.0,2006.0,0,NaN,NaN,NaN
1112,1113,a,c,9260.0,NaN,NaN,0,NaN,NaN,NaN
1113,1114,a,c,870.0,NaN,NaN,0,NaN,NaN,NaN


## 2.6 Leyenda.

In [62]:
tiendas.columns

Index(['Store', 'StoreType', 'Assortment', 'CompetitionDistance',
       'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2',
       'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval'],
      dtype='object')

##### StoreType: categoría que indica el tipo de tienda (a, b, c, d)
##### Assortment:  a = básico, b = extra, c = extedido
##### CompetitionDistance (en metros): distancia a la tienda de la competencia más cercana
##### CompetitionOpenSince [Mes/Año]:  fecha en que abrió la competencia
##### Promo2: Promo2 es una promoción continuada y consecutiva en algunas tiendas (0 = la tienda no participa, 1 = la tienda participa)
##### Promo2Since [Año/Semana]: fecha en la que la tienda empieza a participar en la Promo2
##### PromoInterval: describe los intervalos consecutivos donde la Promo2 empieza, indicando los meses en los que empieza la misma. P.e. "Feb,May,Aug,Nov" significa que cada nueva ronda de promoción empieza en Febrero, Mayo, Agosto, Noviembre de cualquier año de esa tienda


## 2.7 Primeras descripciones y conclusiones.

In [65]:
tiendas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1115 entries, 0 to 1114
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Store                      1115 non-null   int64  
 1   StoreType                  1115 non-null   object 
 2   Assortment                 1115 non-null   object 
 3   CompetitionDistance        1112 non-null   float64
 4   CompetitionOpenSinceMonth  761 non-null    float64
 5   CompetitionOpenSinceYear   761 non-null    float64
 6   Promo2                     1115 non-null   int64  
 7   Promo2SinceWeek            571 non-null    float64
 8   Promo2SinceYear            571 non-null    float64
 9   PromoInterval              571 non-null    object 
dtypes: float64(5), int64(2), object(3)
memory usage: 87.2+ KB


#### 10 columnas en total, 5 flotantes, 2 enteros y 3 objetos.

In [64]:
tiendas.describe()

,Store,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear
count,1115.00000,1112.000000,761.000000,761.000000,1115.000000,571.000000,571.000000
mean,558.00000,5404.901079,7.224704,2008.668857,0.512108,23.595447,2011.763573
std,322.01708,7663.174720,3.212348,6.195983,0.500078,14.141984,1.674935
min,1.00000,20.000000,1.000000,1900.000000,0.000000,1.000000,2009.000000
25%,279.50000,717.500000,4.000000,2006.000000,0.000000,13.000000,2011.000000
50%,558.00000,2325.000000,8.000000,2010.000000,1.000000,22.000000,2012.000000
75%,836.50000,6882.500000,10.000000,2013.000000,1.000000,37.000000,2013.000000
max,1115.00000,75860.000000,12.000000,2015.000000,1.000000,50.000000,2015.000000


### Conclusiones:
##### 1.- Distancia promedio de la competencia: 5404 metros, competencia a menor distancia: 20 metros, competencia a mayor distancia: 75860 metros. 
##### 2.- En promedio, las competencias abren en julio, y la mayoría lo hace entre abril y mediados de octubre.
##### 3.- En promedio, las tiendas fueron abiertas en 2008 y la mayoría lo hizo entre medidados de 2002 y finales de 2014. Una tienda abrió en 1900.
##### 4.- En promedio, la mitad de las tiendas tienen promo2.
##### 5.- En promedio, las tiendas comenzaron a tener la promo 2 en el año 2011, pero la mayoría la obtuvo entre 2010 y 2013.

# TAREA #3: EXPLORAR EL DATASET

TAREA #3.1: EXPLORAR EL DATASET DE VENTAS

In [24]:
# Veamos si nos faltan datos, ¡esperemos que no sea así!


In [25]:
# Promedio de 600 clientes por día, el máximo es 4500 (¡tenga en cuenta que no podemos ver el valor atípico en 7388!)
# Los datos se distribuyen por igual en varios días de la semana (~ 150000 observaciones x 7 días = ~ 1,1 millones de observaciones)
# Las tiendas están abiertas ~ 80% del tiempo
# Los datos se distribuyen por igual entre todas las tiendas (sin sesgo)
# La promoción # 1 se ejecutó aproximadamente el 40% del tiempo
# Ventas promedio alrededor de 5000-6000 Euros
# Las vacaciones escolares duran alrededor del 18% del tiempo


In [26]:
# ¡Veamos cuántas tiendas están abiertas y cerradas!


In [27]:
# Contemos el número de tiendas que están abiertas y cerradas

In [28]:
# nos quedamos solo con las tiendas abiertas y eliminamos las tiendas cerradas


In [29]:
# Eliminemos la columna open ya que ahora no tiene sentido


In [30]:
# Ventas promedio = 6955 Euros, número promedio de clientes = 762 (ha subido)

TAREA #3.2: EXPLORAR LOS DATOS DE LA INFORMACIÓN DE LAS TIENDAS

In [31]:
# ¡Veamos si falta algún dato en el data frame de información de la tienda!


In [32]:
# Echemos un vistazo a los valores faltantes en la 'CompetitionDistance'
# Solo faltan 3 filas



In [33]:
# Echemos un vistazo a los valores faltantes en el 'CompetitionOpenSinceMonth'
# Faltan muchas filas = 354 (casi un tercio de las 1115 tiendas)


In [34]:
# Parece que si 'promo2' es cero, 'promo2SinceWeek', 'Promo2SinceYear' y la información de 'PromoInterval' se establece en cero
# Hay 354 filas donde 'CompetitionOpenSinceYear' y 'CompetitionOpenSinceMonth' falta
# Establezcamos estos valores en ceros


In [35]:
# Hay 3 filas con valores de 'CompetitionDistance' que faltan, llenémoslas con valores promedio de la columna 'CompetitionDistance'

In [36]:
# la mitad de las tiendas participan en la promoción 2
# la mitad de las tiendas tienen su competencia a una distancia de 0-3000 m (3 kms de distancia)


TAREA #3.3: EXPLOREMOS EL DATASET COMBINADO

In [37]:
# Combinemos ambos data frames en función de 'store'


In [38]:

# los clientes y la promoción se correlacionan positivamente con las ventas
# Promo2 no parece ser efectivo en absoluto

In [39]:

# Los clientes / Promo2 y las ventas están fuertemente correlacionados

In [40]:
# Separemos el año y pongámoslo en una columna separada

In [41]:
# Hagamos lo mismo para el día y el mes


In [42]:
# Echemos un vistazo a las ventas promedio y la cantidad de clientes por mes.
# 'groupby' funciona muy bien al agrupar todos los datos que comparten la misma columna del mes, luego obtener la media de la columna de ventas
# Parece que las ventas y el número de clientes alcanzan su punto máximo en el período de Navidad

In [43]:
# Echemos un vistazo a las ventas y a los clientes por día del mes.
# El número mínimo de clientes suele rondar el día 24 del mes.
# La mayoría de los clientes y las ventas son alrededor del 30 y el 1 del mes


In [44]:
# Hagamos lo mismo para el día de la semana (notemos que 7 = domingo)


# TAREA # 4: ENTENDER LA INTUICIÓN DETRÁS DE FACEBOOK PROPHET

![alt text](https://drive.google.com/uc?id=1zk5cwXQvqci6hOwXoTSU0DY0Dpt1jNF0)

![alt text](https://drive.google.com/uc?id=10HNgyFmM1CG6yzSxpFudunUaHKNV_xBT)

![alt text](https://drive.google.com/uc?id=1BhEzmHYh_HKDWGz4eXP-clF7KAxJ_Od7)

# TAREA # 5: ENTRENAR AL MODELO PARTE A

In [45]:
# import prophet 

# TAREA # 6: ENTRENAR AL MODELO PARTE B

- StateHoliday: indica si el día era festivo o no (a = vacaciones públicas, b = vacaciones de Pascua holiday, c = Navidades, 0 = No era festivo)
- SchoolHoliday: indica si (Store, Date) se ve afectado por el cierre de las escuelas públicas
  






In [46]:
# Obtener todas las fechas relacionadas con las vacaciones escolares

In [47]:
# Obtener todas las fechas correspondientes a los festivos estatales


In [48]:
# Concatenamos las vacaciones escolares y los festivos estatales


In [49]:
# Hagamos predicciones usando días festivos para una tienda específica


# BUEN TRABAJO!!!